In [1]:
import torch
import os
%load_ext autoreload
%autoreload 2


import genesis as gs
import logging
gs.init(logging_level=logging.WARNING, backend=gs.gpu)
from buffer import Buffer
from network import Network
from make_environment import Go2WalkingEnv
from reward import Rewards

reward_fn = Rewards(
    forward_weight=0.2,
    alive_weight=0.5,
    upright_weight=0.3
)
max_steps = 100

env = Go2WalkingEnv(
    num_envs=1,
    device="mps",
    show_viewer=False,
    use_terrain=False,  # Set to True for complex terrain
    episode_length_s=20.0,
    reward_fn=reward_fn
)
env.set_commands(lin_vel_x=1.0, lin_vel_y=0.0, ang_vel_yaw=0.0)
policy = Network(
    num_outputs=env.num_actions,
    num_inputs=env.num_obs,
    gamma=0.99,
    lmbda=0.0,
    epsilon=0.1,
)
buffer = Buffer(
    num_envs=1,
    obs_dim=env.num_obs,
    act_dim=env.num_actions,
    max_length=max_steps,
    device='mps'
)
optim = torch.optim.Adam(policy.parameters(),
                         lr=5e-4,
                         eps=1e-8)

device = torch.device('mps')

checkpoint = torch.load("/Users/felix/PycharmProjects/Genesis-Dog-Walking/test/go2_update_tmp.pt")
policy.load_state_dict(checkpoint["model_state_dict"])
optim.load_state_dict(checkpoint["optimizer_state_dict"])



[I 03/13/26 15:19:20.178 1993608] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout
2026-03-13 15:19:22.535 Python[84382:1993608] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/3y/snyjd7qd59d_gxq1x0y5gywh0000gn/T/org.python.python.savedState


[Genesis] [15:19:23] [WARNING] Viewer option 'n_rendered_envs' is deprecated and will be removed in future release. Please use 'rendered_envs_idx' instead.
[Genesis] [15:19:28] [WARNING] Neutral robot position (qpos0) exceeds joint limits.


In [2]:
import os
import torch

def make_eval_video(
    env,
    policy,
    filename="videos/eval.mp4",
    eval_steps=600,
    fps=50,
    deterministic=True,
):
    os.makedirs(os.path.dirname(filename) or ".", exist_ok=True)

    policy.eval()
    obs = env.reset()

    cam = env.camera
    cam.start_recording()

    episode_reward = 0.0

    with torch.no_grad():
        for step in range(eval_steps):
            if deterministic:
                action_mean, _, _ = policy.forward(obs)
                actions = torch.clamp(action_mean, -1.0, 1.0)
            else:
                actions, _ = policy.get_actions(obs)
                actions = torch.clamp(actions, -1.0, 1.0)

            obs, reward, done, info = env.step(actions)
            episode_reward += reward.mean().item()

            cam.render()

            if done[0].item():
                break

    cam.stop_recording(save_to_filename=filename, fps=fps)
    policy.train()

    return {
        "video_path": filename,
        "episode_reward": episode_reward,
        "steps": step + 1,
    }


In [3]:
make_eval_video(
        env=env,
        policy=policy,
        filename=f"/Users/felix/PycharmProjects/Genesis-Dog-Walking/test/eval_update_tmp.mp4",
        eval_steps=600,
    )

UNSUPPORTED (log once): POSSIBLE ISSUE: unit 4 GLD_TEXTURE_INDEX_CUBE_MAP is unloadable and bound to sampler type (Float) - using zero texture because texture unloadable


{'video_path': '/Users/felix/PycharmProjects/Genesis-Dog-Walking/test/eval_update_tmp.mp4',
 'episode_reward': 281.88302333652973,
 'steps': 600}